# KHÓA LUẬN TỐT NGHIỆP CNTT-KLCN155
## MA TRẬN THỰC NGHIỆM PHÁT HIỆN ĐA ĐỐI TƯỢNG RÁC THẢI (2x4 ABLATION & WBF)
---
**Đề tài:** *Xây dựng hệ thống phát hiện và phân loại đa đối tượng rác thải sinh hoạt trong ảnh chụp thực tế bằng mô hình học sâu nhẹ và kỹ thuật tăng cường dữ liệu*
**GVHD:** ThS. Huỳnh Thị Châu Lan
**Sinh viên thực hiện:** Ngô Thanh Nhân (MSSV: 2001230595), Võ Gia Ninh, Vũ Trường Vinh

### Mục tiêu khoa học:
1. Huấn luyện toàn diện **Ma trận 8 thực nghiệm (2x4)**:
   - **Mô hình chính:** SSDLite320-MobileNetV3 (học chuyển giao 258 tensor keys từ classifier)
   - **Mô hình đối chứng:** YOLOv8n (chuẩn hóa head 10 lớp)
   - **4 chiến lược Augmentation:** `None`, `Geometric`, `Photometric`, `Combined (Copy-Paste)`
2. Tối ưu hóa siêu tham số **Weighted Boxes Fusion (WBF)** bằng Grid Search trên tập Validation.
3. Đánh giá kiểm chứng độc lập trên tập **Blind Test Set** theo chuẩn quốc tế COCO (`mAP@0.5`, `mAP@0.5:0.95`).
4. Xuất bảng số liệu đối sánh Ablation phục vụ Chương 4 Báo cáo khóa luận.

### Bước 1: Kiểm tra GPU và Cấu hình phần cứng

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('CẢNH BÁO: Hãy chuyển Runtime sang GPU (Runtime > Change runtime type > T4 GPU)')

### Bước 2: Tải mã nguồn dự án từ GitHub và cài đặt thư viện

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Nhan-create/CNTT-KLCN155-waste-detection.git'
PROJECT_DIR = Path('/content/CNTT-KLCN155-waste-detection')

if not PROJECT_DIR.exists():
    !git clone $REPO_URL
else:
    %cd $PROJECT_DIR
    !git pull origin main

%cd $PROJECT_DIR
!pip install -q -r requirements.txt
!pip install -q albumentations pycocotools ultralytics
print('✓ Đã chuẩn bị môi trường tại:', os.getcwd())

### Bước 3: Kiểm toán tính toàn vẹn dữ liệu Detection (Zero-Leakage)

In [ ]:
from src.detection.dataset import validate_detection_dataset
report = validate_detection_dataset(Path('configs/detection_dataset.yaml'))
print('Số lượng ảnh theo từng tập:', report.images_by_split)
print('Số lượng bounding boxes theo tập:', report.boxes_by_split)
print('Số lượng boxes theo từng lớp:', report.boxes_by_class)

### Bước 4: Chạy toàn bộ Ma trận Thực nghiệm 2x4 & Đánh giá WBF
> Lệnh dưới đây tự động chạy tuần tự 8 thí nghiệm (SSDLite & YOLOv8n trên 4 biến thể tăng cường).
> Sau khi huấn luyện, script sẽ tự động tìm kiếm siêu tham số WBF tối ưu trên tập Validation và đánh giá trên Blind Test Set.

In [ ]:
# Tham số cấu hình:
# --epochs: 120 (chạy chính thức) hoặc 5-10 (nếu muốn kiểm tra nhanh luồng)
# --patience: 30 (dừng sớm nếu validation mAP không tăng)
# --device: cuda (dùng GPU T4)
!python scripts/run_experiment_matrix.py \
    --epochs 120 \
    --patience 30 \
    --device cuda \
    --seed 42

### Bước 5: Xem bảng kết quả tổng hợp Ablation và Đối sánh WBF

In [ ]:
import json
import pandas as pd

summary_path = Path('artifacts/detection/matrix_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f'Thời điểm hoàn thành (UTC): {summary["timestamp_utc"]}')
    
    if summary.get('evaluations'):
        df_eval = pd.DataFrame(summary['evaluations'])
        cols = ['model_name', 'backend', 'test_mAP50', 'test_mAP50_95', 'precision', 'recall', 'f1', 'latency_p50_ms', 'fps']
        display(df_eval[[c for c in cols if c in df_eval.columns]])
    else:
        print('Chưa có thông tin đánh giá evaluations.')
else:
    print('Chưa tìm thấy tệp matrix_summary.json')

### Bước 6: Đóng gói và tải toàn bộ Trọng số & Báo cáo (.zip)

In [ ]:
!zip -q -r /content/ket_qua_thuc_nghiem_phase2.zip artifacts/detection/
print('✓ Đã nén thành công: /content/ket_qua_thuc_nghiem_phase2.zip')

try:
    from google.colab import files
    files.download('/content/ket_qua_thuc_nghiem_phase2.zip')
except Exception as e:
    print('Bạn có thể tải trực tiếp file ket_qua_thuc_nghiem_phase2.zip từ thư mục Files bên trái.')